# 003 Normalized Candidate Sets

This notebook inspects the normalized POC candidate and review sets produced from the source-specific preprocessing gate. It still does not train, index, or enable answer generation.

The stage promotes clean Stack Exchange POC records into classifier/retrieval candidate pools, preserves security and firmware records as question-only safety/eval fixtures, and keeps ticket datasets in a manual review queue.

## Stage Contract

- Retrieval candidates are POC/share-alike constrained and not commercial-ready.
- Safety fixtures are question-only; security/firmware answer text is not promoted.
- Ticket datasets remain blocked from training until manual PII/license/provenance/schema review.
- `downstream_allowed` flags are part of the artifact contract and must be checked by later notebooks.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

DATA = ROOT / 'data'
OUT = DATA / 'processed' / 'normalized_candidate_sets'
PYTHON = ROOT / '.it_support' / 'Scripts' / 'python.exe'

ROOT

## Regenerate Artifacts

Set `RUN_BUILD = True` only after gate artifacts change. This script reads raw Stack Exchange data plus gate outputs and writes normalized candidate/review artifacts.

In [ ]:
RUN_BUILD = False

cmd = [str(PYTHON if PYTHON.exists() else sys.executable), 'scripts/build_normalized_candidate_sets.py']
print(' '.join(cmd))

if RUN_BUILD:
    completed = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, check=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
else:
    print('Build not rerun. Reading existing artifacts.')

## Summary

In [ ]:
display(Markdown((OUT / 'normalization_summary.md').read_text(encoding='utf-8')))
summary = json.loads((OUT / 'normalization_summary.json').read_text(encoding='utf-8'))
summary['counts']

## Classifier Pool

Question-only pool across all Stack Exchange gated records. This preserves multi-label and safety signal coverage, but later stages must respect each record's `promotion_status` and `downstream_allowed` flags.

In [ ]:
classifier_pool = pd.read_json(OUT / 'stack_exchange_classifier_pool.jsonl', lines=True)
display(classifier_pool['promotion_status'].value_counts().rename_axis('promotion_status').reset_index(name='records'))
display(classifier_pool['primary_domain'].value_counts().rename_axis('primary_domain').reset_index(name='records'))
classifier_pool[['record_id', 'title', 'primary_domain', 'secondary_domains', 'gate_decision', 'promotion_status']].head(10)

## Retrieval Candidates

These are the only records with answer evidence promoted in this stage. They are still POC/share-alike and answer-generation-disabled.

In [ ]:
retrieval = pd.read_json(OUT / 'stack_exchange_retrieval_candidates.jsonl', lines=True)
display(retrieval['gate_decision'].value_counts().rename_axis('gate_decision').reset_index(name='records'))
display(retrieval['primary_domain'].value_counts().rename_axis('primary_domain').reset_index(name='records'))
retrieval[['record_id', 'title', 'primary_domain', 'secondary_domains', 'license', 'commercial_reuse_allowed']].head(10)

## Safety Fixtures

These records are question-only fixtures for security and firmware evaluation. They should help measure false negatives and escalation behavior, not provide procedural answers.

In [ ]:
safety = pd.read_json(OUT / 'stack_exchange_safety_eval_fixtures.jsonl', lines=True)
display(safety['expected_behavior'].value_counts().rename_axis('expected_behavior').reset_index(name='records'))
display(safety['primary_domain'].value_counts().rename_axis('primary_domain').reset_index(name='records'))
safety[['record_id', 'title', 'primary_domain', 'secondary_domains', 'expected_behavior', 'blocked_reason']].head(20)

## Manual Review Queues

These are not promoted. Review them manually before any future normalization.

In [ ]:
stack_review = pd.read_json(OUT / 'stack_exchange_manual_review_queue.jsonl', lines=True)
ticket_review = pd.read_json(OUT / 'ticket_dataset_manual_review_queue.jsonl', lines=True)

display(stack_review['primary_domain'].value_counts().rename_axis('primary_domain').reset_index(name='records'))
display(ticket_review[['dataset_id', 'declared_license', 'declared_pii_removed', 'declared_synthetic', 'pii_signal_count', 'secret_like_signal_count', 'promotion_status']])

## Exit Criteria

Before retrieval/indexing, add a downstream loader that enforces:

- `downstream_allowed.retrieval == True` for retrieval inputs.
- `commercial_mode == False` excludes all POC/share-alike/non-commercial records.
- Safety fixtures are used for eval only.
- Ticket datasets remain blocked until a manual review artifact promotes specific datasets/files.